# Lesson 13 — Cutting Planes

## Learning objectives

1. Derive a Gomory cut from a fractional simplex tableau {cite:p}`Gomory1958`.
2. Recognize Chvátal-Gomory, MIR, lift-and-project, and split cuts.
3. Understand cut management in modern solvers (cut pool, separation, age).
4. Use `discopt` cut callbacks.

## 1. The cutting-plane idea

If the LP relaxation $z_{LP}$ is fractional, find a linear inequality $\alpha^\top x \le \beta$ that:

- holds for every integer-feasible $x$ (valid),
- is *violated* by the current LP optimum $x_{LP}$.

Add it; resolve. Repeat. Each iteration tightens the relaxation. {cite:p}`Wolsey1998,Conforti2014` are the textbook references.

## 2. Gomory mixed-integer (GMI) cut

Take a row of the optimal LP tableau in which the basic variable $x_i$ has fractional value, with fractional part $f_i = x_i - \lfloor x_i \rfloor \in (0, 1)$. Let $f_{ij}$ be the fractional part of the row's coefficient on non-basic *integer* variable $j$, and $a_{ij}$ the row's coefficient on non-basic *continuous* variable $j$. The **Gomory mixed-integer cut** is

$$
\sum_{\substack{j \in N_I \\ f_{ij} \le f_i}} \frac{f_{ij}}{f_i} x_j \;+\; \sum_{\substack{j \in N_I \\ f_{ij} > f_i}} \frac{1 - f_{ij}}{1 - f_i} x_j \;+\; \sum_{\substack{j \in N_C \\ a_{ij} \ge 0}} \frac{a_{ij}}{f_i} x_j \;-\; \sum_{\substack{j \in N_C \\ a_{ij} < 0}} \frac{a_{ij}}{1 - f_i} x_j \;\ge\; 1.
$$

(Pure-integer Gomory cuts {cite:p}`Gomory1958` are the special case with no continuous variables.) The cut separates the current $x^{LP}$ (where every non-basic variable is at zero, so the LHS is 0) from every integer-feasible point. See {cite:p}`Wolsey1998,Conforti2014,Cornuejols2008` for the derivation.

## 3. Mixed-integer rounding (MIR)

Take a valid inequality with an integer-valued LHS and a *mixed* RHS: integer variables $x_j \in \mathbb{Z}_+$ ($j \in I$), continuous $y_j \ge 0$ ($j \in C$). Let $f_j = a_j - \lfloor a_j \rfloor$ and $f_b = b - \lfloor b \rfloor$. The MIR inequality is

$$\sum_{j \in I} \Big(\lfloor a_j \rfloor + \frac{(f_j - f_b)^+}{1 - f_b}\Big) x_j \;-\; \sum_{\substack{j \in C \\ a_j < 0}} \frac{a_j}{1 - f_b} y_j \;\le\; \lfloor b \rfloor + \frac{f_b}{1 - f_b}\cdot 0$$

with $(z)^+ = \max(z, 0)$. The continuous part is what distinguishes MIR from pure Chvátal–Gomory rounding (which is the integer-only special case). MIR is one of the most effective cut families in modern solvers {cite:p}`Cornuejols2008`.

## 4. Lift-and-project

For 0/1 problems: form the disjunction $x_j = 0 \vee x_j = 1$, take the convex hull of the two LP relaxations, and project onto $x$-space. Yields a polynomial-time procedure for generating cuts {cite:p}`Balas1993`.

## 5. Cut management

Modern MILP solvers add hundreds of cuts per node. They:

- **Pool** cuts that aren't currently violated.
- **Age** them out if not used in N nodes.
- Apply cuts globally (root) vs locally (single subtree).
- Use **strong cuts** (deep, dense) sparingly and **weak cuts** (sparse, cheap) often.

Cut quality is measured by depth: distance from $x_{LP}$ to the cut hyperplane.

In [ ]:
# discopt course compat shim — installs `add_variable / add_variables /
# add_constraint / add_constraints / is_convex` and a friendly `.solve(...)`
# on `discopt.Model`. Run this cell first.
import sys, pathlib
_repo = pathlib.Path.cwd()
while _repo != _repo.parent and not (_repo / "course").is_dir(): _repo = _repo.parent
if str(_repo) not in sys.path: sys.path.insert(0, str(_repo))
from course._compat import *  # noqa: F401,F403


In [ ]:
import numpy as np, discopt as do

# Toy MILP where Gomory cuts close the gap
# max x1 + x2 s.t. 2x1 + x2 <= 5, x1 + 3x2 <= 6, x1,x2 in {0,1,2,...}
m = do.Model("cut_demo")
x = m.add_variables(2, vtype="integer", lb=0, ub=5, name="x")
m.add_constraint(2*x[0] + x[1] <= 5)
m.add_constraint(x[0] + 3*x[1] <= 6)
m.maximize(x[0] + x[1])

r = m.solve(cuts=False, log_relaxation=True)
print("no cuts: nodes =", r.nodes, " z* =", r.objective)
r = m.solve(cuts=True,  log_relaxation=True)
print("with cuts: nodes =", r.nodes, " z* =", r.objective)


## References
{cite:p}`Gomory1958` (original), {cite:p}`Balas1993` (lift-and-project),
{cite:p}`Cornuejols2008` (modern survey),
{cite:p}`Wolsey1998,Conforti2014` (textbooks).